#DAY 4 Databricks Challenge

##Challenges
### 🛠️ Tasks:

1. Convert CSV to Delta format
2. Create Delta tables (SQL and PySpark)
3. Test schema enforcement
4. Handle duplicate inserts

###We will be loading kaggle dataset loaded already into the volume

In [0]:

events = spark.read.csv(\
    "/Volumes/workspace/ecommerce/ecommerce_data",
    header=True,
    inferSchema=True
)
display(events)

##First we create a volume called delta under default

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.delta;


### Task 1. Convert to Delta format while saving

####📌 What happens internally:

####1. Data is stored as Parquet
####2. _delta_log/ is created
####3. Every write is tracked transactionally


In [0]:
events.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/delta/events")


###Verifying the saved data

In [0]:
spark.read.format("delta") \
    .load("/Volumes/workspace/default/delta/events") \
    .show(5)


###Task 2: Create Delta Tables (PySpark & SQL)

####Step 1: Delta formatted table from the dataset loaded

In [0]:
events.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("events_table")

####Step 2: Creating Delta formatted table from existing events_table using SQL

In [0]:
%sql
CREATE TABLE events_delta
USING DELTA
AS
SELECT * FROM events_table


####Verify the created table

In [0]:
%sql
SHOW TABLES;
SELECT COUNT(*) FROM events_delta;


###Task 3: Test Schema Enforcement (VERY IMPORTANT)
####Delta does NOT allow incompatible schemas by default.
####This protects your data from corruption.

Delta performs pre-write validation:

- Reads the Delta table schema from _delta_log
- Compares it with incoming DataFrame schema
- Detects: Column mismatch and Type mismatch
- Aborts the write
- Throws an exception


In [0]:
try:
    wrong_schema = spark.createDataFrame(
        [("a", "b", "c")],
        ["x", "y", "z"]
    )

    wrong_schema.write \
        .format("delta") \
        .mode("append") \
        .save("/Volumes/workspace/default/delta/events")

except Exception as e:
    print(f"Schema enforcement triggered: {e}")


###Task 4: Handle Duplicate Inserts (MOST IMPORTANT)

Duplicates happen when:

- Jobs rerun
- Late-arriving data
- Retries occur

Delta gives MERGE (UPSERT) to fix this.

####Two not good approaches
#####A) Duplicate Inserts possible

events.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("events_table")
  
#####B)Only removes duplicates while appending and not on data that comes after appending

deduped_events = events.dropDuplicates(
    ["event_time", "user_id", "product_id", "event_type"]
)

deduped_events.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("events_table")


####Best Approach - Using UPSERT(UPDATE or INSERT)

#### Step 1: Create incoming data (Example)

In [0]:
new_events = events.limit(10)
new_events.createOrReplaceTempView("new_events")
display(new_events)


#####Step 2: MERGE (UPSERT Logic)

In [0]:
%sql
MERGE INTO events_table t
USING new_events s
ON t.event_time = s.event_time
AND t.user_id = s.user_id
AND t.product_id = s.product_id
AND t.event_type = s.event_type

WHEN MATCHED THEN
  UPDATE SET *

WHEN NOT MATCHED THEN
  INSERT *

## **For me more such learning and insights in**
- ### [LinkedIn](https://www.linkedin.com/in/ilakkiyan-av/) 
- ### [Youtube](https://www.youtube.com/@ilakkiyanav) 